<a href="https://colab.research.google.com/github/isaac-sun/20NEWS-FL/blob/main/colab_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
# 20 Newsgroups Federated Learning — Free-Rider Attack Detection

**Full DistilBERT Fine-Tuning + Per-Class Shapley Value Detection**

4 experiments: Baseline → DFR → SDFR → AFR

## 1. Setup

Clone the repository. If you opened via the **Open in Colab** badge, the repo is already cloned — just run this cell to confirm.

In [ ]:
import os, sys

REPO_NAME = "20NEWS-FL"
GITHUB_USER = "isaac-sun"

if os.path.basename(os.getcwd()) == REPO_NAME:
    print(f"Already in {REPO_NAME}/ — skipping clone.")
elif os.path.exists(REPO_NAME):
    print(f"{REPO_NAME}/ exists, pulling latest...")
    %cd {REPO_NAME}
    !git pull
else:
    print("Cloning repository...")
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git
    %cd {REPO_NAME}

print(f"Working directory: {os.getcwd()}")

## 2. Install Dependencies

In [ ]:
# Keep Colab's CUDA-enabled PyTorch; replacing it can break GPU support.
!pip install -q -r requirements-colab.txt
print("Dependencies installed; Colab PyTorch was preserved.")

## 3. Hugging Face Token (Recommended)

Authenticated requests get higher rate limits and faster downloads.

1. Go to https://huggingface.co/settings/tokens → create a **Read** token
2. In Colab: click 🔑 **Secrets** → add `HF_TOKEN` → toggle ON

Your token is encrypted and never committed to GitHub.

In [ ]:
import os
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        print("HF_TOKEN loaded from Colab Secrets.")
    else:
        print("No HF_TOKEN set — running unauthenticated (slower but works).")
except ImportError:
    print("Not in Colab — using OS environment variables.")

## 4. Verify GPU

In [ ]:
import torch
print(f"PyTorch:  {torch.__version__}")
print(f"CUDA:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:      {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM:     {vram:.1f} GB")
else:
    raise RuntimeError("CUDA is unavailable. Select Runtime > Change runtime type > GPU.")

## 5. Run All Experiments

This runs Baseline, DFR, SDFR, and AFR for 50 rounds each. Runtime depends strongly on the assigned GPU and Shapley evaluation cost. Keep this tab connected until all four experiments finish.

In [ ]:
!FORCE_DEVICE=cuda python -u main.py --require-cuda --gpu-profile auto --results-dir results

## 6. Results

In [ ]:
import pandas as pd
import os
from IPython.display import Image, display as ipy_display

xlsx_path = "results/experiment_results.xlsx"
if os.path.exists(xlsx_path):
    df = pd.read_excel(xlsx_path, sheet_name="experiment_summary")
    display(df)
else:
    print("Results file not found. Did the experiment finish?")

plots_dir = "results/plots"
if os.path.isdir(plots_dir):
    for f in sorted(os.listdir(plots_dir)):
        if f.endswith(".png"):
            print(f"\n### {f}")
            ipy_display(Image(os.path.join(plots_dir, f)))

## 7. Save to Google Drive and Download

In [ ]:
import datetime
import os
import shutil
from google.colab import drive, files

archive = shutil.make_archive("results", "zip", "results")
drive.mount("/content/drive")
drive_dir = "/content/drive/MyDrive/20NEWS-FL-results"
os.makedirs(drive_dir, exist_ok=True)
timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
drive_path = os.path.join(drive_dir, f"results-{timestamp}.zip")
shutil.copy2(archive, drive_path)
print(f"Saved to {drive_path}")
files.download(archive)